## Botmaker: One-click generation of domain expert intelligent agents

### Shared Setup
Load the local OpenScholar_Retriever model and helper utilities used in multiple tasks.

In [1]:
import os
import json
import re
from pathlib import Path
from typing import Iterable, List, Dict, Any
import gc

# Workaround for OpenMP duplicate runtime crash on Windows.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import torch
from transformers import AutoModel, AutoTokenizer

BASE_DIR = Path.cwd()
PASSAGES_DIR = BASE_DIR / "OpenScholar-DataStore-V3" / "passages"
RETRIEVER_DIR = BASE_DIR / "OpenScholar_Retriever"

assert PASSAGES_DIR.exists(), f"Passages directory not found: {PASSAGES_DIR}"
assert RETRIEVER_DIR.exists(), f"Reranker directory not found: {RETRIEVER_DIR}"

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        try:
            _ = torch.empty((1,), device="cuda")
            return torch.device("cuda")
        except Exception as exc:
            print(f"CUDA init failed, falling back to CPU: {exc}")
    return torch.device("cpu")

DEVICE = pick_device()
torch.set_num_threads(1)

# Load tokenizer and encoder from the local OpenScholar_Retriever folder.
TOKENIZER = AutoTokenizer.from_pretrained(str(RETRIEVER_DIR), local_files_only=True)
MODEL = AutoModel.from_pretrained(str(RETRIEVER_DIR), local_files_only=True)
MODEL.eval()

try:
    MODEL.to(DEVICE)
    if DEVICE.type == "cuda":
        MODEL = MODEL.half()
except Exception as exc:
    print(f"Model to {DEVICE} failed, using CPU: {exc}")
    DEVICE = torch.device("cpu")
    MODEL.to(DEVICE)

def mean_pooling(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

@torch.inference_mode()
def encode_texts(texts: List[str], batch_size: int = 32) -> torch.Tensor:
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        batch = TOKENIZER(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(DEVICE)
        outputs = MODEL(**batch)
        embeddings = mean_pooling(outputs.last_hidden_state, batch["attention_mask"])
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0) if all_embeddings else torch.empty((0, 0))

c:\Users\cszmli\AppData\Local\miniconda3\envs\huggingface-download\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 39417.89it/s]
[transformers] BertModel LOAD REPORT from: d:\Projects\BotMaker\OpenScholar_Retriever
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Task 1 — Extract first paragraphs and embeddings

In [2]:
try:
    import orjson  # type: ignore
    _HAS_ORJSON = True
except Exception:
    orjson = None
    _HAS_ORJSON = False

def iter_jsonl_files(folder: Path) -> Iterable[Path]:
    return sorted(p for p in folder.glob("*.jsonl") if p.is_file())

def _process_file_streaming(
    file_path: Path,
    out_path: Path,
    use_orjson: bool,
 ) -> int:
    if use_orjson and orjson is not None:
        json_loads = orjson.loads
        decode_needed = False
    else:
        json_loads = json.loads
        decode_needed = True
    seen: set[str] = set()
    count = 0
    with file_path.open("rb") as f, out_path.open("w", encoding="utf-8") as out:
        for line in f:
            if not line.strip():
                continue
            try:
                payload = line.decode("utf-8") if decode_needed else line
                record = json_loads(payload)
            except Exception:
                continue
            raw_id = record.get("raw_id")
            if raw_id is None or raw_id in seen:
                continue
            seen.add(raw_id)
            out.write(
                json.dumps({"raw_id": raw_id, "text": record.get("text", "")}, ensure_ascii=False)
                + "\n"
            )
            count += 1
    return count

def extract_first_paragraphs(
    passages_dir: Path,
    use_orjson: bool = _HAS_ORJSON,
 ) -> List[Path]:
    files = list(iter_jsonl_files(passages_dir))
    if not files:
        return []
    tmp_dir = BASE_DIR / "tmp_first_paragraphs"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    tmp_files: List[Path] = []
    for file_path in files:
        tmp_path = tmp_dir / f"{file_path.stem}.first_paragraphs.jsonl"
        if tmp_path.exists():
            print(f"Skip existing output -> {tmp_path}")
            tmp_files.append(tmp_path)
            continue
        count = _process_file_streaming(
            file_path=file_path,
            out_path=tmp_path,
            use_orjson=use_orjson,
        )
        print(f"Saved {count} first paragraphs -> {tmp_path}")
        tmp_files.append(tmp_path)
    return tmp_files

def first_sentence(text: str) -> str:
    if not text:
        return ""
    return text[:300].strip()  

def write_jsonl(path: Path, records: Iterable[Dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def iter_jsonl_records(path: Path) -> Iterable[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                yield json.loads(line)
            except Exception:
                continue

# # 1) Extract first paragraphs based on first raw_id appearance (streaming, low memory)
tmp_first_paragraph_files = extract_first_paragraphs(PASSAGES_DIR)
print(f"Saved {len(tmp_first_paragraph_files)} per-file first paragraph outputs")

# 2) Extract first sentence from each first paragraph (per-file)
tmp_sentence_dir = BASE_DIR / "tmp_first_sentences"
tmp_sentence_dir.mkdir(parents=True, exist_ok=True)
for tmp_path in tmp_first_paragraph_files:
    out_path = tmp_sentence_dir / tmp_path.name.replace(".first_paragraphs", ".first_sentences")
    with out_path.open("w", encoding="utf-8") as f:
        count = 0
        for rec in iter_jsonl_records(tmp_path):
            text = rec.get("text", "")
            if not text:
                continue
            payload = {"raw_id": rec.get("raw_id"), "first_sentence": first_sentence(text)}
            f.write(json.dumps(payload, ensure_ascii=False) + "\n")
            count += 1
    print(f"Saved {count} first sentences -> {out_path}")

Skip existing output -> d:\Projects\BotMaker\tmp_first_paragraphs\raw_passages-0-of-16.first_paragraphs.jsonl
Skip existing output -> d:\Projects\BotMaker\tmp_first_paragraphs\raw_passages-1-of-16.first_paragraphs.jsonl
Skip existing output -> d:\Projects\BotMaker\tmp_first_paragraphs\raw_passages-10-of-16.first_paragraphs.jsonl
Skip existing output -> d:\Projects\BotMaker\tmp_first_paragraphs\raw_passages-11-of-16.first_paragraphs.jsonl
Skip existing output -> d:\Projects\BotMaker\tmp_first_paragraphs\raw_passages-12-of-16.first_paragraphs.jsonl
Skip existing output -> d:\Projects\BotMaker\tmp_first_paragraphs\raw_passages-13-of-16.first_paragraphs.jsonl
Skip existing output -> d:\Projects\BotMaker\tmp_first_paragraphs\raw_passages-14-of-16.first_paragraphs.jsonl
Skip existing output -> d:\Projects\BotMaker\tmp_first_paragraphs\raw_passages-15-of-16.first_paragraphs.jsonl
Skip existing output -> d:\Projects\BotMaker\tmp_first_paragraphs\raw_passages-2-of-16.first_paragraphs.jsonl
Skip

In [4]:
# Check raw_id uniqueness across tmp_first_sentences jsonl files
tmp_sentence_dir = BASE_DIR / "tmp_first_sentences"
all_raw_ids: set[str] = set()
dup_raw_ids: set[str] = set()
file_counts: Dict[str, int] = {}
if not tmp_sentence_dir.exists():
    print(f"Directory not found: {tmp_sentence_dir}")
else:
    for path in sorted(tmp_sentence_dir.glob("*.jsonl")):
        file_seen: set[str] = set()
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                except Exception:
                    continue
                raw_id = rec.get("raw_id")
                if raw_id is None:
                    continue
                if raw_id in file_seen:
                    dup_raw_ids.add(raw_id)
                else:
                    file_seen.add(raw_id)
                if raw_id in all_raw_ids:
                    dup_raw_ids.add(raw_id)
                else:
                    all_raw_ids.add(raw_id)
        file_counts[path.name] = len(file_seen)

    print(f"Total unique raw_id: {len(all_raw_ids)}")
    print(f"Duplicate raw_id count: {len(dup_raw_ids)}")
    if dup_raw_ids:
        sample = sorted(list(dup_raw_ids))[:20]
        print(f"Sample duplicates: {sample}")
    print(f"Files checked: {len(file_counts)}")

Total unique raw_id: 45200821
Duplicate raw_id count: 0
Files checked: 16


In [33]:
# Build IndexFlat from large PKL files by streaming vectors
import pickle
from pathlib import Path
from typing import Iterator, Iterable

import numpy as np
import faiss


def _yield_vectors_from_object(obj: object) -> Iterator[np.ndarray]:
    if isinstance(obj, np.ndarray):
        if obj.ndim == 1:
            yield obj
        else:
            for row in obj:
                yield row
        return
    if isinstance(obj, (list, tuple)):
        for item in obj:
            if isinstance(item, np.ndarray):
                if item.ndim == 1:
                    yield item
                else:
                    for row in item:
                        yield row
            else:
                yield np.asarray(item)
        return
    yield np.asarray(obj)

def iter_pkl_vectors(pkl_path: Path) -> Iterator[np.ndarray]:
    # Supports files written as multiple pickle dumps or a single object.
    with pkl_path.open("rb") as f:
        unpickler = pickle.Unpickler(f)
        while True:
            try:
                obj = unpickler.load()
            except EOFError:
                break
            for vec in _yield_vectors_from_object(obj):
                yield vec

def _make_hnsw_ip_index(d: int, m: int) -> faiss.IndexHNSWFlat:
    try:
        return faiss.IndexHNSWFlat(d, m, faiss.METRIC_INNER_PRODUCT)
    except TypeError:
        index = faiss.IndexHNSWFlat(d, m)
        index.metric_type = faiss.METRIC_INNER_PRODUCT
        return index


def _l2_normalize_rows(arr: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-9, None)
    return arr / norms


def build_indexflat_from_pkl(
    pkl_path: Path,
    out_dir: Path,
    batch_size: int = 4096,
    metric: str = "ip",
    log_every_batches: int = 10,
) -> Path:
    if metric == "l2":
        out_path = out_dir / f"{pkl_path.stem}.indexflat_l2.faiss"
    elif metric == "ip":
        out_path = out_dir / f"{pkl_path.stem}.indexflat.faiss"
    else:
        raise ValueError(f"metric must be 'l2' or 'ip', got {metric!r}")
    if out_path.exists():
        print(f"Skip existing index -> {out_path}")
        return out_path
    vectors_iter = iter_pkl_vectors(pkl_path)
    first = next(vectors_iter, None)
    if first is None:
        raise ValueError(f"No vectors found in {pkl_path}")
    first = np.asarray(first, dtype="float32")
    if first.ndim != 1:
        raise ValueError(f"Expected 1D vector, got shape {first.shape}")
    d = int(first.shape[0])
    if metric == "l2":
        index = faiss.IndexFlatL2(d)
    else:
        index = faiss.IndexFlatIP(d)

    def add_batch(batch: Iterable[np.ndarray], metric: str) -> int:
        arr = np.vstack([np.asarray(v, dtype="float32") for v in batch])
        if metric == "ip":
            arr = _l2_normalize_rows(arr)
        arr = np.ascontiguousarray(arr, dtype="float32")
        index.add(arr)
        return arr.shape[0]

    count = 0
    batch = [first]
    batch_id = 0
    for vec in vectors_iter:
        batch.append(vec)
        if len(batch) >= batch_size:
            batch_id += 1
            count += add_batch(batch, metric)
            if batch_id % log_every_batches == 0:
                print(f"[IndexFlat {metric}] Batches: {batch_id} | Vectors: {count}")
            batch.clear()
    if batch:
        batch_id += 1
        count += add_batch(batch, metric)
        print(f"[IndexFlat {metric}] Batches: {batch_id} | Vectors: {count}")

    faiss.write_index(index, str(out_path))
    print(f"[IndexFlat {metric}] Saved {count} vectors -> {out_path}")
    return out_path


def build_hnsw_index_from_pkl(
    pkl_path: Path,
    out_dir: Path,
    batch_size: int = 4096,
    hnsw_m: int = 32,
    ef_construction: int = 40,
    ef_search: int = 64,
    log_every_batches: int = 10,
) -> Path:
    out_path = out_dir / f"{pkl_path.stem}.hnsw_ip.faiss"
    if out_path.exists():
        print(f"Skip existing index -> {out_path}")
        return out_path
    vectors_iter = iter_pkl_vectors(pkl_path)
    first = next(vectors_iter, None)
    if first is None:
        raise ValueError(f"No vectors found in {pkl_path}")
    first = np.asarray(first, dtype="float32")
    if first.ndim != 1:
        raise ValueError(f"Expected 1D vector, got shape {first.shape}")
    d = int(first.shape[0])

    index = _make_hnsw_ip_index(d, hnsw_m)
    index.hnsw.efConstruction = ef_construction
    index.hnsw.efSearch = ef_search

    def add_batch(batch: Iterable[np.ndarray]) -> int:
        arr = np.vstack([np.asarray(v, dtype="float32") for v in batch])
        arr = _l2_normalize_rows(arr)
        arr = np.ascontiguousarray(arr, dtype="float32")
        index.add(arr)
        return arr.shape[0]

    count = 0
    batch = [first]
    batch_id = 0
    for vec in vectors_iter:
        batch.append(vec)
        if len(batch) >= batch_size:
            batch_id += 1
            count += add_batch(batch)
            if batch_id % log_every_batches == 0:
                print(f"[HNSW IP] Batches: {batch_id} | Vectors: {count}")
            batch.clear()
    if batch:
        batch_id += 1
        count += add_batch(batch)
        print(f"[HNSW IP] Batches: {batch_id} | Vectors: {count}")

    faiss.write_index(index, str(out_path))
    print(f"[HNSW IP] Saved {count} vectors -> {out_path}")
    return out_path

In [ ]:
PKL_DIR = Path(r"C:\Projects\Biomaker_tmpdata\tmp_sentence_embeddings")
OUT_DIR = BASE_DIR / "tmp_sentence_indexes"
OUT_DIR.mkdir(parents=True, exist_ok=True)
pkl_files = sorted(PKL_DIR.glob("*.pkl"))
if not pkl_files:
    print(f"No PKL files found in {PKL_DIR}")
else:
    print(f"Found {len(pkl_files)} PKL files in {PKL_DIR}")
    for pkl_path in pkl_files:
        build_indexflat_from_pkl(pkl_path, OUT_DIR, batch_size=4096, metric="ip")

Found 16 PKL files in C:\Projects\Biomaker_tmpdata\tmp_sentence_embeddings
Skip existing index -> d:\Projects\BotMaker\tmp_sentence_indexes\raw_passages-0-of-16.first_sentences.embeddings.f16.indexflat.faiss
Batches: 10 | Vectors: 40960
Batches: 20 | Vectors: 81920
Batches: 30 | Vectors: 122880
Batches: 40 | Vectors: 163840
Batches: 50 | Vectors: 204800
Batches: 60 | Vectors: 245760
Batches: 70 | Vectors: 286720
Batches: 80 | Vectors: 327680
Batches: 90 | Vectors: 368640
Batches: 100 | Vectors: 409600
Batches: 110 | Vectors: 450560
Batches: 120 | Vectors: 491520
Batches: 130 | Vectors: 532480
Batches: 140 | Vectors: 573440
Batches: 150 | Vectors: 614400
Batches: 160 | Vectors: 655360
Batches: 170 | Vectors: 696320
Batches: 180 | Vectors: 737280
Batches: 190 | Vectors: 778240
Batches: 200 | Vectors: 819200
Batches: 210 | Vectors: 860160
Batches: 220 | Vectors: 901120
Batches: 230 | Vectors: 942080
Batches: 240 | Vectors: 983040
Batches: 250 | Vectors: 1024000
Batches: 260 | Vectors: 106

In [34]:
# Index class + multi-index search (includes raw_id, supports batch queries)
import heapq
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Tuple, Dict
from dataclasses import dataclass
import time
import gc

import pickle
from pathlib import Path

import numpy as np
import faiss

@dataclass
class SearchHit:
    score: float
    local_idx: int
    raw_id: str
    index_path: Path
    meta_path: Path

class FaissIndex:
    def __init__(self, index_path: Path, meta_path: Path, metric: str = "ip", omp_threads: int = 8, load_on_init: bool = True) -> None:
        self.index_path = index_path
        self.meta_path = meta_path
        self.metric = metric
        self.omp_threads = omp_threads
        self.use_max = metric.lower() == "ip"
        self.index: faiss.Index | None = None
        if load_on_init:
            if not self.index_path.exists():
                raise FileNotFoundError(f"Index file not found: {self.index_path}")
            t0 = time.perf_counter()
            self.index = faiss.read_index(str(self.index_path))
            faiss.omp_set_num_threads(self.omp_threads)
            t1 = time.perf_counter()
            print(f"Loaded index {self.index_path.name}: {t1 - t0:.4f}s | ntotal={self.index.ntotal}")

    def unload(self) -> None:
        if self.index is not None:
            del self.index
            self.index = None
            gc.collect()

    def _load_raw_ids(self, hits: List[int]) -> Dict[int, str]:
        if not hits or not self.meta_path.exists():
            return {}
        max_idx = max(hits)
        raw_id_map: Dict[int, str] = {}
        with self.meta_path.open("r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i > max_idx:
                    break
                if i in hits:
                    try:
                        raw_id_map[i] = json.loads(line).get("raw_id", "")
                    except Exception:
                        raw_id_map[i] = ""
        return raw_id_map

    def search(self, query: np.ndarray, per_index_k: int) -> List[List[SearchHit]]:
        if not self.index_path.exists():
            print(f"Index not found: {self.index_path}")
            return [[] for _ in range(query.shape[0])]
        # ensure index is loaded (lazy-load if not loaded at construction)
        if self.index is None:
            t0 = time.perf_counter()
            self.index = faiss.read_index(str(self.index_path))
            faiss.omp_set_num_threads(self.omp_threads)
            t1 = time.perf_counter()
            print(f"Lazy-loaded index {self.index_path.name}: {t1 - t0:.4f}s | ntotal={self.index.ntotal}")
        q = np.ascontiguousarray(query, dtype="float32")
        scores, indices = self.index.search(q, per_index_k)
        hits_all = sorted(set(indices.reshape(-1).tolist()))
        raw_ids = self._load_raw_ids(hits_all)
        results: List[List[SearchHit]] = [[] for _ in range(query.shape[0])]
        for qi in range(query.shape[0]):
            for score, local_idx in zip(scores[qi].tolist(), indices[qi].tolist()):
                raw_id = raw_ids.get(local_idx, "")
                results[qi].append(SearchHit(score, local_idx, raw_id, self.index_path, self.meta_path))
        return results

def _normalize_query(query_emb: np.ndarray) -> np.ndarray:
    if query_emb.ndim == 1:
        query = query_emb.reshape(1, -1)
    else:
        query = query_emb
    return np.ascontiguousarray(query, dtype="float32")

def multi_index_search(
    indexes: List[FaissIndex],
    query_emb: np.ndarray,
    per_index_k: int,
    final_k: int,
    metric: str = "ip",
    unload_after_index: bool = True,
    workers: int = 1,
 ) -> List[List[SearchHit]]:
    """Search across multiple `FaissIndex` instances.

    Args:
        indexes: list of FaissIndex objects to query.
        query_emb: query embeddings (1D or 2D numpy array).
        per_index_k: number of top results to fetch from each index.
        final_k: final number of top results to keep per query.
        metric: "ip" for inner product (higher is better) or "l2" for distance (lower is better).
        unload_after_index: if True, call `unload()` on each FaissIndex after using it to free memory.
        workers: number of indexes to search concurrently. Defaults to 1 for sequential search.
    """
    if workers < 1:
        raise ValueError("workers must be >= 1")

    query = _normalize_query(query_emb)
    use_max = metric.lower() == "ip"
    best: List[List[Tuple[float, int, str, Path, Path]]] = [[] for _ in range(query.shape[0])]

    def search_one(idx: FaissIndex) -> List[List[SearchHit]]:
        try:
            return idx.search(query, per_index_k)
        finally:
            if unload_after_index:
                try:
                    idx.unload()
                    gc.collect()
                except Exception:
                    pass

    def add_results(per_index_results: List[List[SearchHit]]) -> None:
        for qi, hits in enumerate(per_index_results):
            for hit in hits:
                item = (hit.score, hit.local_idx, hit.raw_id, hit.index_path, hit.meta_path)
                heap = best[qi]
                if len(heap) < final_k:
                    heapq.heappush(heap, item if use_max else (-hit.score, hit.local_idx, hit.raw_id, hit.index_path, hit.meta_path))
                else:
                    if use_max:
                        if hit.score > heap[0][0]:
                            heapq.heapreplace(heap, item)
                    else:
                        if -hit.score > heap[0][0]:
                            heapq.heapreplace(heap, (-hit.score, hit.local_idx, hit.raw_id, hit.index_path, hit.meta_path))

    if workers == 1 or len(indexes) <= 1:
        for idx in indexes:
            add_results(search_one(idx))
    else:
        max_workers = min(workers, len(indexes))
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(search_one, idx) for idx in indexes]
            for future in as_completed(futures):
                add_results(future.result())

    results: List[List[SearchHit]] = []
    if use_max:
        for heap in best:
            heap_sorted = sorted(heap, key=lambda x: x[0], reverse=True)
            results.append([SearchHit(*item) for item in heap_sorted])
        return results
    for heap in best:
        heap_sorted = sorted([(-s, i, r, p, m) for s, i, r, p, m in heap], key=lambda x: x[0])
        results.append([SearchHit(*item) for item in heap_sorted])
    return results

In [38]:
# Example usage
# INDEX_DIR = Path(r"C:\Projects\Biomaker_tmpdata\sentence_hnsw_indexes")
# META_DIR = Path(r"C:\Projects\Biomaker_tmpdata\sentence_meta")
# index_paths = sorted(INDEX_DIR.glob("*.faiss"))
# indexes: List[FaissIndex] = []
# for idx_path in index_paths:
#     meta_path = META_DIR / f"{str(idx_path.name).split('.')[0]}.first_sentences.raw_id.jsonl"
#     # load_on_init=True will read the faiss index during FaissIndex(...)
#     indexes.append(FaissIndex(idx_path, meta_path, metric="ip", omp_threads=16, load_on_init=False))

# Example usage
indexes: List[FaissIndex] = []
indexes.append(FaissIndex(Path(r"D:\Projects\BotMaker\microbiome_ecology_test_1\passage_embeddings\passage_index\passage_embeddings.indexflat.faiss"), 
Path(r"D:\Projects\BotMaker\microbiome_ecology_test_1\passage_embeddings\passage_embeddings.meta.jsonl"), 
metric="ip", omp_threads=16, load_on_init=False))


query_texts = [
    "How do antibiotics affect the gut microbiome?",
]
query_emb = encode_texts(query_texts, batch_size=2).numpy().astype("float32", copy=False)
all_results = multi_index_search(indexes, query_emb, per_index_k=5, final_k=5, metric="ip")
for q_idx, results in enumerate(all_results, start=1):
    print(f"\nQuery {q_idx} top results:")
    for rank, hit in enumerate(results, start=1):
        print(
            f"{rank}. score={hit.score:.4f} idx={hit.local_idx} raw_id={hit.raw_id} "
            f"index={hit.index_path.name} meta={hit.meta_path.name}"
        )


Lazy-loaded index passage_embeddings.indexflat.faiss: 0.0902s | ntotal=90952

Query 1 top results:
1. score=0.7528 idx=27254 raw_id=233462057 index=passage_embeddings.indexflat.faiss meta=passage_embeddings.meta.jsonl
2. score=0.7346 idx=75802 raw_id=196663539 index=passage_embeddings.indexflat.faiss meta=passage_embeddings.meta.jsonl
3. score=0.7283 idx=76262 raw_id=218582467 index=passage_embeddings.indexflat.faiss meta=passage_embeddings.meta.jsonl
4. score=0.7128 idx=55621 raw_id=232023522 index=passage_embeddings.indexflat.faiss meta=passage_embeddings.meta.jsonl
5. score=0.6969 idx=70503 raw_id=51599108 index=passage_embeddings.indexflat.faiss meta=passage_embeddings.meta.jsonl


### Task 2 — Semantic Scholar Bulk Search
Retrieve papers from the Semantic Scholar `/paper/search/bulk` endpoint using keyword queries. Results are deduplicated by `paperId` and saved as JSON.

In [1]:
import os
import json
import re
import time
from pathlib import Path
from typing import Optional

import requests
from openai import OpenAI

# ---------------------------------------------------------------------------
# Semantic Scholar query syntax reference (embedded in LLM prompts)
# ---------------------------------------------------------------------------
_QUERY_SYNTAX = """\
Semantic Scholar Boolean query syntax:
- `fish ladder`        -> implicit AND
- `+`                  -> explicit AND
- `|`                  -> OR
- `-`                  -> NOT (negate a term)
- `"..."`              -> exact phrase
- `*`                  -> prefix wildcard (e.g. `bio*`)
- `( )`                -> grouping / precedence
- `~N`                 -> edit distance (e.g. `fish~`, `fish~1`)
- `"..."~N`            -> phrase slop (e.g. `"fish ladder"~3`)
Terms are stemmed (running <-> run). Searches are case-insensitive."""

# ---------------------------------------------------------------------------
# Prompt builders
# ---------------------------------------------------------------------------
def _build_seed_prompt(expert_name: str, expert_description: str) -> str:
    return (
        "You are a literature retrieval specialist. "
        "Generate a SINGLE Semantic Scholar bulk-search query string for the "
        "expert described below.\n"
        "\n"
        f"Expert Name: {expert_name}\n"
        f"Expert Description: {expert_description}\n"
        "\n"
        f"{_QUERY_SYNTAX}\n"
        "\n"
        "Guidelines:\n"
        "1. Cover the core concepts of the expert's domain using OR (|) groups "
        "of synonyms / related terms.\n"
        "2. Link different concept groups with AND (+) so papers must touch "
        "all aspects.\n"
        "3. Use NOT (-) to exclude clearly irrelevant areas.\n"
        "4. Favour high recall: prefer broad OR groups and avoid overly "
        "restrictive AND chains.\n"
        "5. The query MUST be in English.\n"
        "6. Output a JSON object with a single key \"query\" containing the "
        "query string, wrapped in ```json ``` fences.\n"
        "Example output:\n"
        "```json\n"
        "{\"query\": \"...\"}\n"
        "```"
    )


def _build_broaden_prompt(
    expert_name: str,
    expert_description: str,
    previous_queries: list[str],
    last_query_count: int,
    cumulative_total: int,
    ) -> str:
    prev_qs = "\n".join(f"  [{i}] {q}" for i, q in enumerate(previous_queries, 1))
    return (
        "You are a literature retrieval specialist. "
        f"The last query returned {last_query_count} new unique papers. "
        f"Cumulative total across all queries so far: {cumulative_total} - "
        "fewer than desired. "
        "Generate a BROADER query to capture more relevant literature.\n"
        "\n"
        f"Expert Name: {expert_name}\n"
        f"Expert Description: {expert_description}\n"
        f"Queries Already Used:\n{prev_qs}\n"
        f"New Papers from Last Query: {last_query_count}\n"
        f"Cumulative Papers So Far: {cumulative_total}\n"
        "\n"
        f"{_QUERY_SYNTAX}\n"
        "\n"
        "Broadening tactics (apply one or more, but keep it conservative):\n"
        "1. Add 1-2 close synonyms inside existing OR (|) groups.\n"
        "2. Prefer keeping at least two AND (+) concept groups.\n"
        "3. Avoid turning the whole query into a single large OR chain.\n"
        "4. Keep any existing NOT (-) exclusions unless clearly over-restrictive.\n"
        "5. Use wildcards (*) sparingly and only on domain-specific terms.\n"
        "\n"
        "Rules:\n"
        "1. The new query MUST differ from all previously used queries.\n"
        "2. The query MUST be in English.\n"
        "3. Avoid broadening so much that results exceed the bulk-search limit.\n"
        "4. Output a JSON object with a single key \"query\" containing the "
        "query string, wrapped in ```json ``` fences.\n"
        "Example output:\n"
        "```json\n"
        "{\"query\": \"...\"}\n"
        "```"
    )


def _build_narrow_prompt(
    expert_name: str,
    expert_description: str,
    previous_queries: list[str],
    too_many_queries: list[str],
    last_error: str,
    ) -> str:
    prev_qs = "\n".join(f"  [{i}] {q}" for i, q in enumerate(previous_queries, 1))
    too_many = "\n".join(f"  [{i}] {q}" for i, q in enumerate(too_many_queries, 1))
    return (
        "You are a literature retrieval specialist. "
        "The last query was rejected because it returned too many hits."
        "Generate a NARROWER query that stays under the bulk-search limit.\n"
        "\n"
        f"Expert Name: {expert_name}\n"
        f"Expert Description: {expert_description}\n"
        f"Last Error: {last_error}\n"
        f"Queries Already Used:\n{prev_qs}\n"
        f"Too-Broad Queries:\n{too_many}\n"
        "\n"
        f"{_QUERY_SYNTAX}\n"
        "\n"
        "Narrowing tactics (apply one or more):\n"
        "1. Add 1-2 AND (+) concept groups to require multiple aspects.\n"
        "2. Remove the broadest OR terms or generic words.\n"
        "3. Replace broad terms with specific phrases in quotes.\n"
        "4. Avoid huge OR-only structures.\n"
        "\n"
        "Rules:\n"
        "1. The new query MUST differ from all previously used queries.\n"
        "2. The query MUST be in English.\n"
        "3. Output a JSON object with a single key \"query\" containing the "
        "query string, wrapped in ```json ``` fences.\n"
        "Example output:\n"
        "```json\n"
        "{\"query\": \"...\"}\n"
        "```"
    )

# ---------------------------------------------------------------------------
# LLM query generator
# ---------------------------------------------------------------------------
_JSON_FENCE_RE = re.compile(r"```json\s*(.*?)\s*```", re.DOTALL)

def _call_llm_for_query(
    client: OpenAI,
    model: str,
    prompt: str,
    ) -> str:
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that generates Semantic Scholar queries."},
            {"role": "user", "content": prompt}],
        temperature=0.8,
        extra_body={"thinking": {"type": "disabled"}}  # disable any "thinking..." intermediate responses
    )
    raw = resp.choices[0].message.content.strip()
    
    # Try to extract JSON from ```json ... ``` fences
    m = _JSON_FENCE_RE.search(raw)
    if m:
        try:
            obj = json.loads(m.group(1))
            if isinstance(obj, dict) and "query" in obj:
                return str(obj["query"]).strip()
        except (json.JSONDecodeError, TypeError, KeyError):
            pass

    # Fallback: treat the whole response as a bare JSON object
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict) and "query" in obj:
            return str(obj["query"]).strip()
    except (json.JSONDecodeError, TypeError, KeyError):
        raise ValueError(f"LLM response does not contain a valid JSON query: {raw}")


class TooManyHitsError(RuntimeError):
    def __init__(self, query: str, error_msg: str) -> None:
        super().__init__(error_msg)
        self.query = query
        self.error_msg = error_msg


def _is_too_many_hits(error_msg: str) -> bool:
    msg = error_msg.lower()
    return "too many hits" in msg and "10000000" in msg


def _generate_query(
    client: OpenAI,
    model: str,
    expert_name: str,
    expert_description: str,
    previous_queries: Optional[list[str]] = None,
    last_query_count: int = 0,
    cumulative_total: int = 0,
    ) -> str:
    if previous_queries is None:
        prompt = _build_seed_prompt(expert_name, expert_description)
    else:
        prompt = _build_broaden_prompt(
            expert_name, expert_description, previous_queries,
            last_query_count, cumulative_total,
        )
    return _call_llm_for_query(client, model, prompt)


def _generate_narrow_query(
    client: OpenAI,
    model: str,
    expert_name: str,
    expert_description: str,
    previous_queries: list[str],
    too_many_queries: list[str],
    last_error: str,
    ) -> str:
    prompt = _build_narrow_prompt(
        expert_name, expert_description, previous_queries, too_many_queries, last_error
    )
    return _call_llm_for_query(client, model, prompt)

# ---------------------------------------------------------------------------
# Core search helper (single query -> all pages, deduplicated)
# ---------------------------------------------------------------------------
def _load_seen_ids_from_jsonl(path: Path) -> set[str]:
    seen: set[str] = set()
    if not path.exists():
        return seen
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            paper_id = rec.get("paperId")
            if paper_id:
                seen.add(str(paper_id))
    return seen

def _fetch_all_for_query(
    keyword: str,
    s2_headers: dict,
    seen_ids: set[str],
    out_path: Path,
    ) -> int:
    """Paginate through ALL results for *keyword*, skip already-seen paperIds.

    New papers are appended to *out_path* as JSONL records.
    """
    base_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
    params: dict = {
        "query": keyword,
        "fields": "paperId,title,abstract,year,authors",
        "publicationDateOrYear": ":2022-12",
        "sort": "publicationDate:desc",
    }
    new_count = 0
    page = 0
    stale_streak = 0
    with out_path.open("a", encoding="utf-8") as out:
        while True:
            resp = requests.get(base_url, params=params, headers=s2_headers, timeout=60)
            if resp.status_code == 429:
                wait = 2 ** min(page, 8) + (time.monotonic() % 1)
                print(f"  Rate-limited, waiting {wait:.1f}s ...")
                time.sleep(wait)
                continue
            body = {}
            try:
                body = resp.json()
            except Exception:
                body = {}
            error_msg = body.get("error", "")
            if error_msg:
                print(f"  Body error received: {error_msg}")
                if _is_too_many_hits(error_msg):
                    raise TooManyHitsError(keyword, error_msg)
            resp.raise_for_status()
            if not body:
                body = resp.json()
            papers = body.get("data", [])
            added_this_page = 0
            for paper in papers:
                pid = paper.get("paperId")
                if pid and pid not in seen_ids:
                    seen_ids.add(str(pid))
                    paper.pop("abstract", None)
                    out.write(json.dumps(paper, ensure_ascii=False) + "\n")
                    new_count += 1
                    added_this_page += 1
            page += 1
            print(
                f"  Page {page}: {len(papers)} returned, "
                f"{added_this_page} new (total unique: {len(seen_ids)})"
            )
            if added_this_page == 0:
                stale_streak += 1
                if stale_streak >= 2:
                    print("  No new papers for 2 consecutive pages - stopping.")
                    break
            else:
                stale_streak = 0
            if "token" not in body:
                break
            params["token"] = body["token"]
            time.sleep(1)

    return new_count

# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------
def search_papers(
    expert_name: str,
    expert_description: str,
    output_path: str = ".",
    start_query: Optional[str] = None,
    *,
    model: str = "gpt-4.1",
    min_papers_per_query: int = 100_000,
    max_papers_per_query: int = 1_000_000,
    max_queries: int = 5,
    ) -> str:
    """Iteratively search Semantic Scholar with LLM-generated queries.

    1. Generates a seed query from *expert_name* + *expert_description* via an
       OpenAI chat model.
    2. Fetches ALL matching papers (full pagination).
    3. If the **cumulative** total of unique papers is still below
       *min_papers_per_query*, the LLM generates a broader query and the
       loop continues.
    4. Stops when the cumulative total reaches *min_papers_per_query*,
       *max_papers_per_query*, or *max_queries* is exhausted.

    Results are saved to ``output_path / expert_name / search_paper.jsonl``.

    Environment variables expected:
    - ``SEMANTIC_SCHOLAR_API_KEY`` (optional, but recommended)
    - ``OPENAI_API_KEY`` (required for LLM query generation)
    - ``OPENAI_API_BASE_URL`` (optional, for custom OpenAI-compatible endpoints)

    Args:
        expert_name: Short name for the domain expert (used as folder name).
        expert_description: Natural-language description of the expert's domain.
        output_path: Base directory for output.
        model: OpenAI chat model name.
        min_papers_per_query: Minimum cumulative papers to collect before
            stopping.
        max_papers_per_query: Hard upper bound - stop immediately if the
            cumulative total reaches this number.
        max_queries: Upper bound on the number of broadening attempts.

    Returns:
        Absolute path to ``search_paper.jsonl``.
    """
    out_dir = Path(output_path) / expert_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / "search_paper.jsonl"

    # --- API clients & headers -------------------------------------------------
    openai_client = OpenAI(
        api_key=os.environ.get("OPENAI_API_KEY"),
        base_url=os.environ.get("OPENAI_API_BASE_URL"),
    )

    s2_headers: dict = {}
    s2_key = os.environ.get("SEMANTIC_SCHOLAR_API_KEY")
    if s2_key:
        s2_headers["x-api-key"] = s2_key

    # --- Iterative search ------------------------------------------------------
    seen_ids: set[str] = set()
    too_many_queries: list[str] = []
    if out_file.exists():
        seen_ids.update(_load_seen_ids_from_jsonl(out_file))
        print(f"Loaded {len(seen_ids)} existing paperId(s) from {out_file}")
    else:
        print(f"Starting new jsonl output -> {out_file}")
    queries_used: list[str] = []
    last_query_new_count: int = 0

    for attempt in range(max_queries):
        # 1. Generate query
        if attempt == 0:
            print(f"--- Generating seed query for {expert_name} ---")
        else:
            print(
                f"--- Broadening query (attempt {attempt + 1}/{max_queries}) "
                f"- cumulative {len(seen_ids)} papers so far ---"
            )
        if start_query and attempt == 0:
            current_query = start_query.strip()
            print(f"Using provided start query: {current_query}")
        else:
            current_query = _generate_query(
                client=openai_client,
                model=model,
                expert_name=expert_name,
                expert_description=expert_description,
                previous_queries=queries_used if queries_used else None,
                last_query_count=last_query_new_count,
                cumulative_total=len(seen_ids),
            )
        if not current_query:
            print("ERROR: LLM returned empty query - skipping this attempt.")
            continue
        queries_used.append(current_query)
        print(f"Query: {current_query}")

        # 2. Fetch all pages
        try:
            new_count = _fetch_all_for_query(
                keyword=current_query,
                s2_headers=s2_headers,
                seen_ids=seen_ids,
                out_path=out_file,
            )
        except TooManyHitsError as exc:
            too_many_queries.append(exc.query)
            print(f"Query too broad: {exc.error_msg}")
            narrowed_query = _generate_narrow_query(
                client=openai_client,
                model=model,
                expert_name=expert_name,
                expert_description=expert_description,
                previous_queries=queries_used,
                too_many_queries=too_many_queries,
                last_error=exc.error_msg,
            )
            if not narrowed_query:
                print("ERROR: LLM returned empty narrowed query - skipping.")
                continue
            queries_used.append(narrowed_query)
            print(f"Narrowed query: {narrowed_query}")
            try:
                new_count = _fetch_all_for_query(
                    keyword=narrowed_query,
                    s2_headers=s2_headers,
                    seen_ids=seen_ids,
                    out_path=out_file,
                )
            except TooManyHitsError as exc:
                too_many_queries.append(exc.query)
                print(f"Narrowed query too broad: {exc.error_msg}")
                break
        last_query_new_count = new_count
        print(
            f"Query added {new_count} new papers "
            f"(cumulative: {len(seen_ids)})"
        )

        # 3. Decide whether to stop or broaden
        if len(seen_ids) >= max_papers_per_query:
            print(
                f"Cumulative {len(seen_ids)} >= {max_papers_per_query} "
                "- stopping."
            )
            break
        if new_count == 0:
            print("Broadened query added zero new papers - stopping.")
        if len(seen_ids) < min_papers_per_query:
            print(
                f"Cumulative {len(seen_ids)} < {min_papers_per_query} "
                "- will broaden."
            )
    print(
        f"\nDone - {len(seen_ids)} unique papers from "
        f"{len(queries_used)} query(s):"
    )
    for i, q in enumerate(queries_used, 1):
        print(f"  [{i}] {q}")
    if too_many_queries:
        print("Too-broad queries (hit limit):")
        for i, q in enumerate(too_many_queries, 1):
            print(f"  [{i}] {q}")
    print(f"Saved to {out_file}")
    return str(out_file.resolve())

In [3]:
# Example: microbiome community ecology expert
os.environ["OPENAI_API_KEY"] = "sk-dce4013929a04466bcebb8e100b91e78"
os.environ["OPENAI_API_BASE_URL"] = "https://api.deepseek.com"  # or your custom endpoint
os.environ["SEMANTIC_SCHOLAR_API_KEY"] = "s2k-awJuRspBcGliFyEAbaFjzwTDIfRppVAj1mYgRLQn"
out = search_papers(
    expert_name="microbiome_ecology_copy",
    expert_description="The expert is responsible for interpreting the biological implications of microbiome data from the perspective of community ecology. After the pipeline generates species annotations and abundance tables, this role needs to assess the rationality of community structure, identify key species, explain the ecological significance of inter‑species relationships, and provide causal reasoning at the microbiological level for functional annotation results.",
    output_path=".",
    model="deepseek-v4-pro",
    max_queries=10,
)
print(f"Output file: {out}")

Loaded 114 existing paperId(s) from microbiome_ecology_copy\search_paper.jsonl
--- Generating seed query for microbiome_ecology_copy ---
Query: (microbiome | microbiota | "microbial community" | "microbial ecology") + ("community structure" | "community assembly" | "co-occurrence" | "species interaction" | "interspecies interaction" | "ecological network" | "keystone species" | "community dynamics" | "beta diversity" | "alpha diversity" | "ecological succession" | "niche differentiation" | "functional redundancy" | "community stability" | "community resilience") + (metagenom* | metatranscriptom* | 16S | "amplicon sequencing" | "shotgun sequencing" | "functional annotation" | "functional profile" | "functional potential" | "metabolic pathway" | "gene abundance") + (interpret* | inference | mechanism* | causality | "causal reasoning" | "biological significance" | "ecological significance" | rationale | implication*) -("drug delivery" | "pharmacokinetic" | "clinical trial" | "tumor microe

### Task 3

In [16]:
import json
import pickle
from pathlib import Path
from typing import List

def process_search_papers(
    dir_path: str,
    mode: str = "encode",
    *,
    batch_size: int = 512,
    indexes: list[FaissIndex] = [],
    embeddings_filename: str = "title_embeddings.pkl",
    per_index_k: int = 5,
    final_k: int = 1,
    metric: str = "ip",
    omp_threads: int = 8,
    unload_after_index = True,
    workers: int = 1,
):
    """支持双功能切换：向量生成与保存 / 相似检索与结果输出。
    """
    base = Path(dir_path)
    jsonl_path = base / "search_paper.jsonl"
    if not jsonl_path.exists():
        raise FileNotFoundError(f"search_paper.jsonl not found in {dir_path}")

    # =========================================================================
    # Mode 1: 向量生成与保存
    # =========================================================================
    if mode == "encode":
        titles: list[str] = []
        with jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                rec = json.loads(line)
                title = (rec.get("title") or "").strip()
                if title:
                    titles.append(title)

        print(f"Loaded {len(titles)} titles from {jsonl_path}")

        embeddings = encode_texts(titles, batch_size=batch_size)
        print(f"Encoded embeddings shape: {embeddings.shape}")

        out_path = base / embeddings_filename
        with out_path.open("wb") as f:
            pickle.dump(embeddings, f)
        print(f"Saved embeddings -> {out_path}")

        return str(out_path.resolve())

    # =========================================================================
    # Mode 2: 相似检索与结果输出
    # =========================================================================
    if mode == "search":
        emb_path = base / embeddings_filename
        if not emb_path.exists():
            raise FileNotFoundError(f"Embeddings file not found: {emb_path}")
        with emb_path.open("rb") as f:
            embeddings = pickle.load(f)
        if isinstance(embeddings, torch.Tensor):
            embeddings = embeddings.numpy()
        embeddings = np.asarray(embeddings, dtype="float32")
        print(f"Loaded embeddings shape: {embeddings.shape}")

        print(f"Loaded {len(indexes)} FAISS indexes")
        all_results = multi_index_search(
            indexes, embeddings,
            per_index_k=per_index_k, final_k=final_k, metric=metric,
            unload_after_index=unload_after_index,
            workers=workers,
        )

        # 构建输出: {索引名.split('.')[0]: [raw_id, ...]}
        # split('.')[0] → raw_passages-0-of-16
        output: dict[str, list[str]] = {}
        for qi, hits in enumerate(all_results):
            for hit in hits:
                idx_key = hit.index_path.stem.split(".")[0]
                output.setdefault(idx_key, [])
                if hit.raw_id and hit.raw_id not in output[idx_key]:
                    output[idx_key].append(hit.raw_id)

        out_path = base / "search_results.json"
        with out_path.open("w", encoding="utf-8") as f:
            json.dump(output, f, ensure_ascii=False, indent=2)
        print(f"Saved search results -> {out_path}")

        return output

    raise ValueError(f"Unknown mode: {mode!r}. Use 'encode' or 'search'.")

In [17]:
# Example usage
# 1. 先生成并保存向量嵌入
process_search_papers(
    dir_path=r"D:\Projects\BotMaker\microbiome_ecology_copy",
    mode="encode",
    batch_size=256
)

Loaded 10428 titles from D:\Projects\BotMaker\microbiome_ecology_copy\search_paper.jsonl
Encoded embeddings shape: torch.Size([10428, 768])
Saved embeddings -> D:\Projects\BotMaker\microbiome_ecology_copy\title_embeddings.pkl


'D:\\Projects\\BotMaker\\microbiome_ecology_copy\\title_embeddings.pkl'

In [ ]:
# Example usage
# 2. 再加载向量并进行相似检索
# 注意替换为实际的 FaissIndex 实例列表
INDEX_DIR = Path(r"C:\Projects\Biomaker_tmpdata\tmp_sentence_indexes")
META_DIR = Path(r"C:\Projects\Biomaker_tmpdata\tmp_sentence_meta")
index_paths = sorted(INDEX_DIR.glob("*.faiss"))
indexes: List[FaissIndex] = []
for idx_path in index_paths:
    meta_path = META_DIR / f"{str(idx_path.name).split('.')[0]}.first_sentences.raw_id.jsonl"
    print(f"Preparing FaissIndex for {idx_path.name} with meta {meta_path}")
    # load_on_init=True will read the faiss index during FaissIndex(...)
    indexes.append(FaissIndex(idx_path, meta_path, metric="ip", omp_threads=16, load_on_init=False))
    break  # remove this break to load all indexes
results = process_search_papers(
    dir_path=r"D:\Projects\BotMaker\microbiome_ecology",
    mode="search",
    indexes=indexes,
    per_index_k=5,
    final_k=1,
    metric="ip",
    omp_threads=8,
)

Preparing FaissIndex for raw_passages-0-of-16.first_sentences.embeddings.f16.indexflat.faiss with meta C:\Projects\Biomaker_tmpdata\tmp_sentence_meta\raw_passages-0-of-16.first_sentences.raw_id.jsonl
Loaded embeddings shape: (409, 768)
Loaded 1 FAISS indexes
Lazy-loaded index raw_passages-0-of-16.first_sentences.embeddings.f16.indexflat.faiss: 3.0140s | ntotal=3036556
Saved search results -> D:\Projects\BotMaker\microbiome_ecology\search_results.json


### Task 4 — Extract passages from search_results.json

1. **Build index (one-time)** — scan each `passages/raw_passages-*-of-16.jsonl` shard and write  
   `OpenScholar-DataStore-V3/passage_indexes/<shard>.offsets.jsonl`  
   (`raw_id` → byte offsets of every passage line for that paper).
2. **Extract** — read `search_results.json` per agent dir; for each shard, `seek` only matching lines  
   (falls back to full-file scan if the index file is missing).

CLI: `python passage_index.py build --workers 4`

In [27]:
import json
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from passage_index import (
    DEFAULT_PASSAGE_INDEX_DIR,
    DEFAULT_PASSAGES_DIR,
    build_all_passage_raw_id_indexes,
    build_passage_raw_id_index,
    extract_passages_from_search_results,
)

PASSAGES_DIR = DEFAULT_PASSAGES_DIR
INDEX_DIR = DEFAULT_PASSAGE_INDEX_DIR

# ---------------------------------------------------------------------------
# Step 1 — Build raw_id -> byte-offset indexes (one-time per datastore)
# ---------------------------------------------------------------------------
# Full build (~16 shards, ~340GB read total; use workers=4–8 on NVMe):
# build_stats = build_all_passage_raw_id_indexes(
#     PASSAGES_DIR,
#     INDEX_DIR,
#     workers=4,
#     force=False,
# )
# print(json.dumps(build_stats, indent=2, ensure_ascii=False))

# Single shard test:
# build_passage_raw_id_index(PASSAGES_DIR, INDEX_DIR, "raw_passages-0-of-16", force=False)

# ---------------------------------------------------------------------------
# Step 2 — Extract passages into <agent_dir>/passages/ (uses index when present)
# ---------------------------------------------------------------------------
stats = extract_passages_from_search_results(
    target_root=r"D:\Projects\BotMaker\microbiome_ecology_copy",
    passages_dir=PASSAGES_DIR,
    index_dir=INDEX_DIR,
    use_index=True,
    workers=4,
    overwrite=False,
)
print(json.dumps(stats, indent=2, ensure_ascii=False))

Found 1 search_results.json file(s)
Passage index dir: D:\Projects\BotMaker\OpenScholar-DataStore-V3\passage_indexes (exists=True)
  raw_passages-1-of-16: 7030 lines written (index)
  raw_passages-11-of-16: 5831 lines written (index)
  raw_passages-10-of-16: 5997 lines written (index)
  raw_passages-0-of-16: 5726 lines written (index)
  raw_passages-12-of-16: 7421 lines written (index)
  raw_passages-13-of-16: 5511 lines written (index)
  raw_passages-14-of-16: 5052 lines written (index)
  raw_passages-15-of-16: 5063 lines written (index)
  raw_passages-2-of-16: 5708 lines written (index)
  raw_passages-3-of-16: 6526 lines written (index)
  raw_passages-4-of-16: 6591 lines written (index)
  raw_passages-5-of-16: 4589 lines written (index)
  raw_passages-9-of-16: 5135 lines written (index)
  raw_passages-6-of-16: 4354 lines written (index)
  raw_passages-7-of-16: 3644 lines written (index)
  raw_passages-8-of-16: 6774 lines written (index)
Processed D:\Projects\BotMaker\microbiome_ecolo

In [30]:
# Task 5 — Extract paragraph embeddings selected in Task 4, then build a paper-level IndexFlat
import json
import pickle
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Iterable

import numpy as np

PICKLED_VECTOR_BYTES = 1664


def _find_chunk_embedding_pkl(embeddings_dir: Path, chunk_name: str) -> Path:
    candidates = [
        embeddings_dir / f"{chunk_name}.pkl",
        embeddings_dir / f"{chunk_name}.embeddings.pkl",
        embeddings_dir / f"{chunk_name}.embeddings.f16.pkl",
        embeddings_dir / f"{chunk_name}.paragraph_embeddings.pkl",
        embeddings_dir / f"{chunk_name}.paragraphs.embeddings.pkl",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    matches = sorted(embeddings_dir.glob(f"{chunk_name}*.pkl"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"No embedding pkl found for chunk {chunk_name!r} in {embeddings_dir}")


def _load_vector_at_index(f, paragraph_idx: int, record_size_bytes: int) -> np.ndarray:
    offset = paragraph_idx * record_size_bytes
    f.seek(offset)
    payload = f.read(record_size_bytes)
    if len(payload) != record_size_bytes:
        raise EOFError(
            f"Cannot read vector at index {paragraph_idx}: "
            f"wanted {record_size_bytes} bytes, got {len(payload)} bytes"
        )
    return np.asarray(pickle.loads(payload))


def _iter_jsonl_records(path: Path) -> Iterable[dict]:
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                yield json.loads(line)
            except Exception:
                continue


def _extract_chunk_embeddings(
    passages_path: Path,
    embeddings_dir: Path,
    record_size_bytes: int,
) -> tuple[str, list[np.ndarray], list[dict], dict[str, int], list[str]]:
    chunk_name = passages_path.stem
    vectors: list[np.ndarray] = []
    metas: list[dict] = []
    messages: list[str] = []
    missing = 0
    skipped = 0

    try:
        source_pkl = _find_chunk_embedding_pkl(embeddings_dir, chunk_name)
    except FileNotFoundError as exc:
        missing = sum(1 for _ in _iter_jsonl_records(passages_path))
        return chunk_name, vectors, metas, {"saved": 0, "missing": missing, "skipped": 0}, [f"  {exc}"]

    with source_pkl.open("rb") as fin:
        for rec in _iter_jsonl_records(passages_path):
            paragraph_id = rec.get("id")
            if paragraph_id is None:
                skipped += 1
                continue

            try:
                paragraph_idx = int(paragraph_id)
                vec = _load_vector_at_index(fin, paragraph_idx, record_size_bytes)
            except Exception as exc:
                skipped += 1
                messages.append(f"  Skip {chunk_name} id={paragraph_id}: {exc}")
                continue

            vectors.append(vec)
            metas.append(
                {
                    "chunk": chunk_name,
                    "text": rec.get("text"),
                    "id": paragraph_idx,
                    "raw_id": rec.get("raw_id"),
                    "source_pkl": str(source_pkl),
                }
            )

    messages.append(f"  {chunk_name}: {len(vectors)} embeddings extracted")
    return chunk_name, vectors, metas, {"saved": len(vectors), "missing": missing, "skipped": skipped}, messages


def _write_extracted_embeddings(
    chunk_results: list[tuple[str, list[np.ndarray], list[dict], dict[str, int], list[str]]],
    out_pkl: Path,
    out_meta: Path,
) -> None:
    with out_pkl.open("wb") as fout, out_meta.open("w", encoding="utf-8") as meta_out:
        for _, vectors, metas, _, _ in chunk_results:
            for vec, meta in zip(vectors, metas):
                pickle.dump(vec, fout, protocol=pickle.HIGHEST_PROTOCOL)
                meta_out.write(json.dumps(meta, ensure_ascii=False) + "\n")


def extract_embeddings_for_passages_dir(
    passages_dir: Path,
    embeddings_dir: Path,
    out_pkl: Path,
    out_meta: Path,
    record_size_bytes: int = PICKLED_VECTOR_BYTES,
    overwrite: bool = False,
    workers: int = 1,
) -> dict[str, int]:
    if out_pkl.exists() and out_meta.exists() and not overwrite:
        meta_count = sum(1 for _ in _iter_jsonl_records(out_meta))
        print(f"Skip existing extracted embeddings -> {out_pkl}")
        return {"saved": meta_count, "missing": 0, "skipped": 0}

    out_pkl.parent.mkdir(parents=True, exist_ok=True)
    out_meta.parent.mkdir(parents=True, exist_ok=True)

    chunk_files = sorted(passages_dir.glob("*.jsonl"))
    workers = max(1, int(workers))
    chunk_results: list[tuple[str, list[np.ndarray], list[dict], dict[str, int], list[str]] | None] = [None] * len(chunk_files)

    if workers > 1:
        with ProcessPoolExecutor(max_workers=workers) as executor:
            futures = {
                executor.submit(_extract_chunk_embeddings, path, embeddings_dir, record_size_bytes): idx
                for idx, path in enumerate(chunk_files)
            }
            for future in as_completed(futures):
                idx = futures[future]
                result = future.result()
                chunk_results[idx] = result
                for msg in result[4]:
                    print(msg)
    else:
        for idx, path in enumerate(chunk_files):
            result = _extract_chunk_embeddings(path, embeddings_dir, record_size_bytes)
            chunk_results[idx] = result
            for msg in result[4]:
                print(msg)

    ordered_results = [result for result in chunk_results if result is not None]
    saved = sum(result[3]["saved"] for result in ordered_results)
    missing = sum(result[3]["missing"] for result in ordered_results)
    skipped = sum(result[3]["skipped"] for result in ordered_results)

    _write_extracted_embeddings(ordered_results, out_pkl, out_meta)
    print(f"Wrote {saved} extracted embeddings -> {out_pkl}")
    print(f"Wrote {saved} metadata rows -> {out_meta}")

    return {"saved": saved, "missing": missing, "skipped": skipped}


def extract_passage_embeddings_and_build_indexes(
    target_root: str | Path,
    embeddings_dir: str | Path,
    record_size_bytes: int = PICKLED_VECTOR_BYTES,
    batch_size: int = 4096,
    metric: str = "ip",
    overwrite_embeddings: bool = False,
    workers: int = 1,
) -> dict:
    if "build_indexflat_from_pkl" not in globals():
        raise RuntimeError("Please run the cell that defines build_indexflat_from_pkl first.")

    target_root = Path(target_root)
    embeddings_dir = Path(embeddings_dir)
    if not target_root.is_dir():
        raise FileNotFoundError(f"Target root not found: {target_root}")
    if not embeddings_dir.is_dir():
        raise FileNotFoundError(f"Embedding pkl directory not found: {embeddings_dir}")

    search_result_paths = sorted(target_root.rglob("search_results.json"))
    if not search_result_paths:
        print(f"No search_results.json found under {target_root}")
        return {}

    stats: dict[str, dict] = {}
    for sr_path in search_result_paths:
        parent_dir = sr_path.parent
        passages_dir = parent_dir / "passages"
        if not passages_dir.is_dir():
            print(f"No passages directory found beside {sr_path}; run Task 4 first")
            continue

        embedding_dir = parent_dir / "passage_embeddings"
        index_dir = embedding_dir / "passage_index"
        out_pkl = embedding_dir / "passage_embeddings.pkl"
        out_meta = embedding_dir / "passage_embeddings.meta.jsonl"

        print(f"Processing {parent_dir}")
        extract_stat = extract_embeddings_for_passages_dir(
            passages_dir=passages_dir,
            embeddings_dir=embeddings_dir,
            out_pkl=out_pkl,
            out_meta=out_meta,
            record_size_bytes=record_size_bytes,
            overwrite=overwrite_embeddings,
            workers=workers,
        )

        if extract_stat["saved"] == 0:
            print(f"  No embeddings extracted for {parent_dir}; index skipped")
            stats[str(parent_dir.resolve())] = {**extract_stat, "index": None}
            continue

        index_dir.mkdir(parents=True, exist_ok=True)
        index_path = build_indexflat_from_pkl(out_pkl, index_dir, batch_size=batch_size, metric=metric)
        stats[str(parent_dir.resolve())] = {**extract_stat, "index": str(index_path)}

    print(json.dumps(stats, indent=2, ensure_ascii=False))
    return stats

In [31]:
# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------
PARAGRAPH_EMBEDDINGS_PKL_DIR = Path(r"C:\Projects\Biomaker_tmpdata\passage_embeddings")
TARGET_ROOT = Path(r"D:\Projects\BotMaker\microbiome_ecology_test_1")
t0 = time.perf_counter()
passage_embedding_stats = extract_passage_embeddings_and_build_indexes(
    target_root=TARGET_ROOT,
    embeddings_dir=PARAGRAPH_EMBEDDINGS_PKL_DIR,
    record_size_bytes=PICKLED_VECTOR_BYTES,
    batch_size=4096,
    metric="ip",
    overwrite_embeddings=True,
    workers=1,
)
t1 = time.perf_counter()
print(f"Time consume: {t1-t0:.2f} seconds")

Processing D:\Projects\BotMaker\microbiome_ecology_test_1
  raw_passages-0-of-16: 5726 embeddings extracted
  raw_passages-1-of-16: 7030 embeddings extracted
  raw_passages-10-of-16: 5997 embeddings extracted
  raw_passages-11-of-16: 5831 embeddings extracted
  raw_passages-12-of-16: 7421 embeddings extracted
  raw_passages-13-of-16: 5511 embeddings extracted
  raw_passages-14-of-16: 5052 embeddings extracted
  raw_passages-15-of-16: 5063 embeddings extracted
  raw_passages-2-of-16: 5708 embeddings extracted
  raw_passages-3-of-16: 6526 embeddings extracted
  raw_passages-4-of-16: 6591 embeddings extracted
  raw_passages-5-of-16: 4589 embeddings extracted
  raw_passages-6-of-16: 4354 embeddings extracted
  raw_passages-7-of-16: 3644 embeddings extracted
  raw_passages-8-of-16: 6774 embeddings extracted
  raw_passages-9-of-16: 5135 embeddings extracted
Wrote 90952 extracted embeddings -> D:\Projects\BotMaker\microbiome_ecology_test_1\passage_embeddings\passage_embeddings.pkl
Wrote 90952

In [10]:
# 检查前 1000 个 key 中的 size 是不是都是 1664
for key in list(A_data.keys())[:1000]:
    if A_data[key]["size"] != 1664:
        print(f"Key {key} has size {len(A_data[key])}")